In [1]:
#Cross attention mechanism

import numpy as np

# Example: tiny embeddings for English words
# Each word is represented by a 3-dimensional vector
english_words = {
    "is": np.array([1, 0, 0]),
    "sleeping": np.array([0, 1, 0])
}

# Stack them as Key (K) and Value (V) matrices
K = np.stack([english_words["is"], english_words["sleeping"]])  # shape (2,3)
V = np.stack([english_words["is"], english_words["sleeping"]])  # same as values

# Query (Q) for the French word we want to generate ("dort")
# Let's say it slightly corresponds to "sleeping"
Q = np.array([0, 0.8, 0.2])  # shape (3,) simplified

# Step 1: Compute attention scores (Q · K^T)
scores = Q @ K.T  # matrix multiplication
print("Raw Scores:", scores)

# Step 2: Apply softmax to get attention weights
exp_scores = np.exp(scores)
attention_weights = exp_scores / np.sum(exp_scores)
print("Attention Weights:", attention_weights)

# Step 3: Weighted sum of Values
output = attention_weights @ V
print("Cross-Attention Output:", output)

Raw Scores: [0.  0.8]
Attention Weights: [0.31002552 0.68997448]
Cross-Attention Output: [0.31002552 0.68997448 0.        ]


In [3]:
#Multi headed mechanism

import numpy as np

# --- Step 1: Input word embeddings (3 words, 4 dimensions each) ---
X = np.array([
    [1, 0, 1, 0],  # Word 1
    [0, 1, 0, 1],  # Word 2
    [1, 1, 0, 0]   # Word 3
])

# --- Step 2: Define 2 attention heads (project X to Q, K, V for each head) ---
# Random simple projection matrices (for demonstration)
W1_Q = np.random.rand(4, 2)
W1_K = np.random.rand(4, 2)
W1_V = np.random.rand(4, 2)

W2_Q = np.random.rand(4, 2)
W2_K = np.random.rand(4, 2)
W2_V = np.random.rand(4, 2)

# --- Step 3: Compute Q, K, V for each head ---
Q1, K1, V1 = X @ W1_Q, X @ W1_K, X @ W1_V
Q2, K2, V2 = X @ W2_Q, X @ W2_K, X @ W2_V

# --- Step 4: Attention function ---
def attention(Q, K, V):
    # Compute scores
    scores = Q @ K.T / np.sqrt(Q.shape[1])
    # Softmax
    exp_scores = np.exp(scores - np.max(scores, axis=1, keepdims=True))  # stability
    weights = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
    # Weighted sum
    return weights @ V

# --- Step 5: Compute attention output for each head ---
head1 = attention(Q1, K1, V1)
head2 = attention(Q2, K2, V2)

# --- Step 6: Concatenate heads and project (final output) ---
multi_head_output = np.concatenate([head1, head2], axis=1)
print("Multi-Head Attention Output:\n", multi_head_output)

Multi-Head Attention Output:
 [[1.68682207 1.06362415 0.73721701 0.81782963]
 [1.68772454 1.06330142 0.78908388 0.8632562 ]
 [1.68720831 1.05973132 0.79077478 0.8656831 ]]


In [4]:
import torch
import torch.nn as nn

# --- Parameters ---
d_model = 8      # embedding size (small for simplicity)
num_heads = 2    # number of attention heads
seq_len = 4      # number of words in sequence
ff_hidden = 16   # hidden size in feed-forward network

# --- Example input: batch of 1 sequence, 4 words, each word 8-dim embedding ---
x = torch.rand(1, seq_len, d_model)  # shape: [batch, seq_len, d_model]

# --- Encoder Block ---
class SimpleEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, ff_hidden):
        super().__init__()
        # Multi-Head Self Attention
        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, batch_first=True)
        # Layer Norm
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        # Feed Forward Network
        self.ff = nn.Sequential(
            nn.Linear(d_model, ff_hidden),
            nn.ReLU(),
            nn.Linear(ff_hidden, d_model)
        )

    def forward(self, x):
        # --- 1. Multi-Head Self-Attention ---
        attn_output, _ = self.attn(x, x, x)  # Q,K,V = x
        # --- 2. Add & Norm ---
        x = self.norm1(x + attn_output)
        # --- 3. Feed Forward ---
        ff_output = self.ff(x)
        # --- 4. Add & Norm ---
        x = self.norm2(x + ff_output)
        return x

# --- Instantiate and run encoder block ---
encoder_block = SimpleEncoderBlock(d_model, num_heads, ff_hidden)
output = encoder_block(x)

print("Input:\n", x)
print("\nOutput:\n", output)

Input:
 tensor([[[0.3902, 0.0724, 0.4743, 0.0824, 0.3123, 0.9959, 0.7744, 0.5910],
         [0.7679, 0.9197, 0.6564, 0.0706, 0.5419, 0.8361, 0.9960, 0.0261],
         [0.7919, 0.9846, 0.5558, 0.1540, 0.2089, 0.9862, 0.4250, 0.9086],
         [0.0948, 0.6886, 0.1331, 0.8373, 0.7371, 0.3566, 0.2462, 0.1201]]])

Output:
 tensor([[[-0.0256, -1.3300, -0.8753, -0.1706,  0.0228,  2.3231,  0.1627,
          -0.1071],
         [ 0.8314,  1.1215, -0.4460, -0.6717,  0.1963,  0.7601,  0.3650,
          -2.1567],
         [ 0.6758,  1.2474, -0.9557, -0.4037, -0.7672,  1.2278, -1.5822,
           0.5579],
         [-0.7976,  0.8028, -1.3528,  1.7922,  0.9186, -0.0903, -0.7234,
          -0.5496]]], grad_fn=<NativeLayerNormBackward0>)


In [5]:
#Tokenization
# --- Example text ---
text = "Running is fun!"

# ----------------------
# 1. Character-level tokenization
char_tokens = list(text)
print("Character-level tokens:", char_tokens)

# ----------------------
# 2. Word-level tokenization (split by spaces)
word_tokens = text.split()
print("Word-level tokens:", word_tokens)

# ----------------------
# 3. Subword-level tokenization using HuggingFace tokenizer (BERT)
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

subword_tokens = tokenizer.tokenize(text)
print("Subword-level tokens:", subword_tokens)

Character-level tokens: ['R', 'u', 'n', 'n', 'i', 'n', 'g', ' ', 'i', 's', ' ', 'f', 'u', 'n', '!']
Word-level tokens: ['Running', 'is', 'fun!']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Subword-level tokens: ['running', 'is', 'fun', '!']


In [6]:
#Small LLM for word findings



pip install torch

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim

# Simple dataset (sentences)
sentences = [
    "i love machine learning",
    "machine learning is fun",
    "i love python",
    "python is powerful"
]

# Create vocabulary
words = list(set(" ".join(sentences).split()))
word_to_ix = {word:i for i,word in enumerate(words)}
ix_to_word = {i:word for word,i in word_to_ix.items()}

vocab_size = len(words)

# Convert sentence into numbers
data = []
for s in sentences:
    w = s.split()
    for i in range(len(w)-1):
        input_word = word_to_ix[w[i]]
        target_word = word_to_ix[w[i+1]]
        data.append((input_word, target_word))

# Simple Neural Network
class SimpleLLM(nn.Module):
    def __init__(self):
        super(SimpleLLM,self).__init__()
        self.embedding = nn.Embedding(vocab_size,10)
        self.linear = nn.Linear(10,vocab_size)

    def forward(self,x):
        x = self.embedding(x)
        x = self.linear(x)
        return x

model = SimpleLLM()

loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.01)

# Training
for epoch in range(200):
    total_loss = 0
    for input_word,target_word in data:

        input_tensor = torch.tensor([input_word])
        target_tensor = torch.tensor([target_word])

        output = model(input_tensor)
        loss = loss_function(output,target_tensor)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

print("Training finished")

Training finished


In [8]:
word = "python"
input_tensor = torch.tensor([word_to_ix[word]])

output = model(input_tensor)
predicted = torch.argmax(output).item()

print("Next word prediction:", ix_to_word[predicted])

Next word prediction: is
